# **8 - Modelagem**

In [ ]:
# separar x e y
X = df_modeld.drop(columns=['TARGET'])
y = df_modeld['TARGET']

In [ ]:
X.shape

(36457, 51)

In [ ]:
y.value_counts(normalize=True) * 100

,proportion
TARGET,
1,56.921853
0,43.078147


## **8.1 - Split treino/teste**

In [ ]:
# stratify=y para garantir que a proporção de bons e maus pagadores seja aproximadamente preservada nos dois conjuntos

from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
# conferindo o balanceamento
y_train.value_counts(normalize=True) * 100

,proportion
TARGET,
1,56.920967
0,43.079033


## **8.2 Modelo 1 - Regressão Logística**

In [ ]:

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

In [ ]:
#escalonamento, SEM fit no conjunto de teste para evitar data leakage
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# como o modelo é desbalanceado pelo TARGET, class_weight='balanced' para atribuir mais peso a classe minoritária


modelo_lr = LogisticRegression(
    class_weight='balanced',
    max_iter=1000,
    random_state=42
)

modelo_lr.fit(
    X_train_scaled,
    y_train
)

LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42)

In [ ]:
#criar variável de predição
y_pred_lr = modelo_lr.predict(X_test_scaled)
y_prob_lr = modelo_lr.predict_proba(X_test_scaled)[:, 1]

In [ ]:
#acurácia do modelo
accuracye_lr = round(accuracy_score(y_test, y_pred_lr), 4)
print("Acurácia:", accuracye_lr)

Acurácia: 0.5281


## 8.2.1 Matriz de Confusão do Modelo 1 - Regressão Logística**

In [ ]:

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    average_precision_score
)

In [ ]:

#Matriz de confusão bons classificados como bons; bons classificados como maus;maus classificados como bons; maus classificados como maus.
confusion_matrix(y_test, y_pred_lr)

array([[1613, 1528],
       [1913, 2238]])

In [ ]:

# linha 1, precisão para maus pagadores
#ROC-AUC: capacidade do modelo de distinguir as duas classes
#PR-AUC: desempenho sobre a classe positiva quando existe desbalanceamento

print(classification_report(y_test, y_pred_lr))

print(
    'ROC-AUC:',
    roc_auc_score(y_test, y_prob_lr)
)

print(
    'PR-AUC:',
    average_precision_score(y_test, y_prob_lr)
)

              precision    recall  f1-score   support

           0       0.46      0.51      0.48      3141
           1       0.59      0.54      0.57      4151

    accuracy                           0.53      7292
   macro avg       0.53      0.53      0.52      7292
weighted avg       0.54      0.53      0.53      7292

ROC-AUC: 0.5420870726079055
PR-AUC: 0.6061445010368753


## **8.3 Modelo 2 - Árvore de Decisão**

In [ ]:
#X31
#Decision Tree não depende da escala dos valores para tomar suas decisões, não precisa rodar o StandardScaler
modelo_tree = DecisionTreeClassifier(
    class_weight='balanced',
    random_state=42,
    max_depth=5
)

modelo_tree.fit(X_train, y_train)

DecisionTreeClassifier(class_weight='balanced', max_depth=5, random_state=42)

In [ ]:
#criar variável de valores previstos
y_pred_tree = modelo_tree.predict(X_test)

y_prob_tree = modelo_tree.predict_proba(X_test)[:, 1]


In [ ]:
#acurácia do modelo
accuracye_tree = round(accuracy_score(y_test, y_pred_tree), 4)
print("Acurácia:", accuracye_tree)

Acurácia: 0.5646


## 8.3.1 Matriz de Confusão do Modelo 2 - Árvore de Decisão**

In [ ]:

#Matriz de confusão bons classificados como bons; bons classificados como maus;maus classificados como bons; maus classificados como maus.
confusion_matrix(y_test, y_pred_tree)

array([[1005, 2136],
       [1039, 3112]])

In [ ]:

# linha 1, precisão para maus pagadores
#ROC-AUC: capacidade do modelo de distinguir as duas classes
#PR-AUC:  desempenho sobre a classe positiva quando existe desbalanceamento

print(classification_report(y_test, y_pred_tree))

print(
    'ROC-AUC:',
    roc_auc_score(y_test, y_prob_tree)
)

print(
    'PR-AUC:',
    average_precision_score(y_test, y_prob_tree)
)

              precision    recall  f1-score   support

           0       0.49      0.32      0.39      3141
           1       0.59      0.75      0.66      4151

    accuracy                           0.56      7292
   macro avg       0.54      0.53      0.52      7292
weighted avg       0.55      0.56      0.54      7292

ROC-AUC: 0.5523334691640186
PR-AUC: 0.6074716833959807


## **8.4 Modelo 3 - Random Forest**

In [ ]:

modelo_rf = RandomForestClassifier(
    n_estimators=300,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

modelo_rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=300, n_jobs=-1,
                       random_state=42)

In [ ]:
#variável de previsões
y_pred_rf = modelo_rf.predict(X_test)

y_prob_rf = modelo_rf.predict_proba(X_test)[:, 1]

In [ ]:
#acurácia do modelo
accuracye = round(accuracy_score(y_test, y_pred_rf), 4)
print("Acurácia:", accuracye)

Acurácia: 0.6872


## 8.4.1 Matriz de Confusão do Modelo 3 - Random Forest**

In [ ]:

#Matriz de confusão bons classificados como bons; bons classificados como maus;maus classificados como bons; maus classificados como maus.
confusion_matrix(y_test, y_pred_rf)

array([[2041, 1100],
       [1181, 2970]])

In [ ]:

# linha 1, precisão para maus pagadores
#ROC-AUC: capacidade do modelo de distinguir as duas classes
#PR-AUC:  desempenho sobre a classe positiva quando existe desbalanceamento

print(classification_report(y_test, y_pred_rf))

print(
    'ROC-AUC:',
    roc_auc_score(y_test, y_prob_rf)
)

print(
    'PR-AUC:',
    average_precision_score(y_test, y_prob_rf)
)

              precision    recall  f1-score   support

           0       0.63      0.65      0.64      3141
           1       0.73      0.72      0.72      4151

    accuracy                           0.69      7292
   macro avg       0.68      0.68      0.68      7292
weighted avg       0.69      0.69      0.69      7292

ROC-AUC: 0.7473074116845528
PR-AUC: 0.7706366806136553


## **8.5 Modelo 4 - Gradient Boosting**






In [ ]:
modelo_gb = HistGradientBoostingClassifier(
    max_iter=200,
    learning_rate=0.05,
    max_leaf_nodes=15,
    random_state=42
)

modelo_gb.fit(X_train, y_train)

HistGradientBoostingClassifier(learning_rate=0.05, max_iter=200,
                               max_leaf_nodes=15, random_state=42)

In [ ]:
#variável de predição
y_pred_gb = modelo_gb.predict(X_test)

y_prob_gb = modelo_gb.predict_proba(X_test)[:, 1]

In [ ]:
#acurácia do modelo
accuracye = round(accuracy_score(y_test, y_pred_gb), 4)
print("Acurácia:", accuracye)

Acurácia: 0.6081


## 8.5.1 Matriz de Confusão do Modelo 3 - Random Forest**

In [ ]:
#Matriz de confusão bons classificados como bons; bons classificados como maus;maus classificados como bons; maus classificados como maus.
confusion_matrix(y_test, y_pred_gb)

array([[ 544, 2597],
       [ 261, 3890]])

In [ ]:
# linha 1, precisão para maus pagadores
#ROC-AUC: capacidade do modelo de distinguir as duas classes
#PR-AUC:  desempenho sobre a classe positiva quando existe desbalanceamento

print(classification_report(y_test, y_pred_gb))

print(
    'ROC-AUC:',
    roc_auc_score(y_test, y_prob_gb)
)

print(
    'PR-AUC:',
    average_precision_score(y_test, y_prob_gb)
)

              precision    recall  f1-score   support

           0       0.68      0.17      0.28      3141
           1       0.60      0.94      0.73      4151

    accuracy                           0.61      7292
   macro avg       0.64      0.56      0.50      7292
weighted avg       0.63      0.61      0.54      7292

ROC-AUC: 0.6244291525630161
PR-AUC: 0.6728016533651353
